In [1]:
import torch
from PIL import Image
from transformers import AutoModel, AutoTokenizer
import torchvision.transforms as T
from torchvision.transforms.functional import InterpolationMode
from tqdm import tqdm
import re
from datasets import load_dataset
import csv
import os
from collections import defaultdict

In [2]:

MODEL_NAME = "OpenGVLab/InternVL2_5-2B"
model = AutoModel.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
).eval().cuda()

/home/system/miniconda3/envs/j/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/home/system/miniconda3/envs/j/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [3]:
vision_encoder = model.vision_model

In [4]:
from PIL import Image
import cv2

image = cv2.imread("left.jpg")
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
image =image/255.0

image = torch.from_numpy(image).permute(-1, 0, 1).unsqueeze(0).to(torch.bfloat16).cuda()
ve_output = vision_encoder(image)

In [9]:
ve_output['pooler_output'].shape #odict_keys(['last_hidden_state', 'pooler_output'])

torch.Size([1, 1024])

In [11]:
model.mlp1

Sequential(
  (0): LayerNorm((4096,), eps=1e-05, elementwise_affine=True)
  (1): Linear(in_features=4096, out_features=2048, bias=True)
  (2): GELU(approximate='none')
  (3): Linear(in_features=2048, out_features=2048, bias=True)
)

In [6]:
for name, param in model.named_parameters():
    print(name)

vision_model.embeddings.class_embedding
vision_model.embeddings.position_embedding
vision_model.embeddings.patch_embedding.weight
vision_model.embeddings.patch_embedding.bias
vision_model.encoder.layers.0.ls1
vision_model.encoder.layers.0.ls2
vision_model.encoder.layers.0.attn.qkv.weight
vision_model.encoder.layers.0.attn.qkv.bias
vision_model.encoder.layers.0.attn.proj.weight
vision_model.encoder.layers.0.attn.proj.bias
vision_model.encoder.layers.0.mlp.fc1.weight
vision_model.encoder.layers.0.mlp.fc1.bias
vision_model.encoder.layers.0.mlp.fc2.weight
vision_model.encoder.layers.0.mlp.fc2.bias
vision_model.encoder.layers.0.norm1.weight
vision_model.encoder.layers.0.norm1.bias
vision_model.encoder.layers.0.norm2.weight
vision_model.encoder.layers.0.norm2.bias
vision_model.encoder.layers.1.ls1
vision_model.encoder.layers.1.ls2
vision_model.encoder.layers.1.attn.qkv.weight
vision_model.encoder.layers.1.attn.qkv.bias
vision_model.encoder.layers.1.attn.proj.weight
vision_model.encoder.layer

In [5]:
from DepthAnythingv2.depth_anything_v2.dpt import DepthAnythingV2
import cv2
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model_configs = {
    'vits': {'encoder': 'vits', 'features': 64, 'out_channels': [48, 96, 192, 384]},
    'vitb': {'encoder': 'vitb', 'features': 128, 'out_channels': [96, 192, 384, 768]},
    'vitl': {'encoder': 'vitl', 'features': 256, 'out_channels': [256, 512, 1024, 1024]},
    'vitg': {'encoder': 'vitg', 'features': 384, 'out_channels': [1536, 1536, 1536, 1536]}
}

encoder = 'vits' # or 'vits', 'vitb', 'vitg'

model = DepthAnythingV2(**model_configs[encoder])
model.load_state_dict(torch.load(f'/media/system/ZERBUIS_EXT_STOR/temp/exp/experiment/depth_tuning/depth_anythingv2_ckpts/depth_anything_v2_vits.pth', map_location='cpu'))
model = model.to(device).eval()

raw_img = cv2.imread('image.png')
depth = model.infer_image(raw_img) 